**In Colab version training model**

!pip install transformers datasets accelerate seqeval

In [ ]:
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

In [ ]:
# Load SROIE Dataset
dataset = load_dataset("jsdnrs/ICDAR2019-SROIE")

train_data = dataset["train"]

print(train_data[0])

In [ ]:
LABEL_LIST = [
    "O",
    "B-TOTAL_AMOUNT", "I-TOTAL_AMOUNT",
    "B-DATE", "I-DATE",
    "B-VENDOR", "I-VENDOR"
]

label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label = {i: l for l, i in label2id.items()}

In [ ]:
# Convert SROIE → NER format
def create_ner_example(example):
    words = example["words"]
    entities = example["entities"]

    labels = ["O"] * len(words)

    def assign_label(entity_value, label_name):
        if entity_value is None:
            return

        tokens = entity_value.split()

        for i in range(len(words)):
            for j in range(len(tokens)):
                if words[i:i+len(tokens)] == tokens:
                    labels[i] = f"B-{label_name}"
                    for k in range(1, len(tokens)):
                        labels[i+k] = f"I-{label_name}"

    assign_label(entities.get("company"), "VENDOR")
    assign_label(entities.get("date"), "DATE")
    assign_label(entities.get("total"), "TOTAL_AMOUNT")

    return {
        "words": words,
        "labels": labels
    }


processed_data = [create_ner_example(x) for x in train_data]

In [ ]:
# Convert SROIE → NER format
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")


def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["words"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    word_ids = tokenized.word_ids()

    labels = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(label2id[example["labels"][word_idx]])
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized["labels"] = labels
    return tokenized


tokenized_data = [tokenize_and_align_labels(x) for x in processed_data[:500]]

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(tokenized_data)

In [ ]:
# Tokenization
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "dslim/bert-base-NER",
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/ner-model",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [ ]:
trainer.save_model("/content/ner-model")
tokenizer.save_pretrained("/content/ner-model")

In [ ]:
import shutil
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

def save_model_to_drive():
    print("Zipping files...")
    # Path to your ner-model directory
    model_dir = "/content/ner-model"
    
    # Path to save zip in Colab temporarily
    temp_zip = "ner-model.zip"
    
    # Create zip of the entire ner-model folder
    shutil.make_archive("ner-model", 'zip', model_dir)
    
    # Destination in Google Drive
    drive_path = "/content/drive/MyDrive/ner-model.zip"
    
    # Copy zip to Google Drive
    shutil.copy(temp_zip, drive_path)
    
    print(f"\n\n✅ Saved Model NER to Google Drive at: {drive_path}")

save_model_to_drive()